# SQLite Database — FinTech 590

Loads pool snapshots and historical data into a local SQLite database.

**Run order:** `defi_pipeline.ipynb` → `historical_data.ipynb` → this notebook.

**Schema:**
```
pools          — one row per pool (current snapshot)
pool_history   — one row per pool per day (historical TVL, APY, IL)
```

## 0. Imports

In [1]:
import sqlite3, pathlib
from datetime import date
import pandas as pd

DB_PATH          = pathlib.Path("data/defi_pools.db")
POOLS_PARQUET    = pathlib.Path("data/top_pools.parquet")
HISTORY_PARQUET  = pathlib.Path("data/pool_history.parquet")

print(f"Database : {DB_PATH}")
print(f"Pools    : {POOLS_PARQUET}")
print(f"History  : {HISTORY_PARQUET}")

Database : data\defi_pools.db
Pools    : data\top_pools.parquet
History  : data\pool_history.parquet


## 1. Create Tables

Uses `CREATE TABLE IF NOT EXISTS` — safe to re-run without wiping data.

In [2]:
con = sqlite3.connect(DB_PATH)
cur = con.cursor()

# pools table — schema managed here; pool_history schema is managed by to_sql (always matches parquet)
cur.execute("""
    CREATE TABLE IF NOT EXISTS pools (
        address            TEXT,
        chain              TEXT,
        token0             TEXT,
        token1             TEXT,
        fee_tier           INTEGER,
        tvl_usd            REAL,
        volume_usd         REAL,
        etherscan_verified INTEGER,
        fetched_at         TEXT,
        PRIMARY KEY (address, chain)
    )
""")
con.commit()
print("Database connection ready.")

# Print current schema
for (t,) in cur.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall():
    print(f"\n  [{t}]")
    for col in cur.execute(f"PRAGMA table_info({t})").fetchall():
        print(f"    {col[1]:20s} {col[2]}" + (" <- PK" if col[5] else ""))

Database connection ready.

  [pool_history]
    address              TEXT <- PK
    date                 TEXT <- PK
    tvl_usd              REAL
    apy                  REAL
    apy_base             REAL
    apy_base_7d          REAL
    il_7d                REAL

  [pools]
    address              TEXT
    chain                TEXT
    token0               TEXT
    token1               TEXT
    fee_tier             INTEGER
    tvl_usd              REAL
    volume_usd           REAL
    etherscan_verified   INTEGER
    fetched_at           TEXT


## 2. Load Pools

Uses `INSERT OR REPLACE` — re-running always reflects the latest snapshot.

In [3]:
pools_df = pd.read_parquet(POOLS_PARQUET)
pools_df["fetched_at"] = str(date.today())

pools_df[["address", "chain", "token0", "token1", "fee_tier",
           "tvl_usd", "volume_usd",
           "etherscan_verified", "fetched_at"]].to_sql(
    "pools", con, if_exists="replace", index=False
)
con.commit()

count = cur.execute("SELECT COUNT(*) FROM pools").fetchone()[0]
print(f"pools table: {count} rows loaded")
pd.read_sql(
    "SELECT chain, address, token0, token1, fee_tier, tvl_usd, etherscan_verified FROM pools ORDER BY chain, tvl_usd DESC",
    con
)

pools table: 47 rows loaded


,chain,address,token0,token1,fee_tier,tvl_usd,etherscan_verified
0,Arbitrum,0x2f5e87C9312fa29aed5c179E456625D79015299c,WBTC,WETH,500,50717129.0,1
1,Arbitrum,0x0E4831319A50228B9e450861297aB92dee15B44F,WBTC,USDC,500,8459632.0,1
2,Arbitrum,0xd13040d4fe917EE704158CfCB3338dCd2838B245,RAIN,WETH,100,3300794.0,1
3,Arbitrum,0xbE3aD6a5669Dc0B8b12FeBC03608860C31E2eef6,USDC,USDT,100,2525341.0,1
4,Arbitrum,0x689C96ceAb93f5E131631D225D75DeA3fD37747E,WBTC,ARB,3000,2161481.0,1
5,Arbitrum,0x80A9ae39310abf666A87C743d6ebBD0E8C42158E,WETH,GMX,10000,2128590.0,1
6,Arbitrum,0x1DFc1054E0e2A10E33C9ca21aaD5AA8A1cCe91e3,HEGIC,WETH,500,1761260.0,1
7,Arbitrum,0x5886e46E6DD497d7501f103a58ff4242bCaa2556,USDC,THBILL,100,1657718.0,1
8,Arbitrum,0x9B42809aaaE8d088eE01FE637E948784730F0386,WBTC,CBBTC,100,1551233.0,1
9,Arbitrum,0xD8D4314f8755259F78cCc2A8902fd92FbBA54877,LON,USDC,3000,1360007.0,1


## 3. Load Historical Data

Uses `INSERT OR IGNORE` — re-running skips existing (address, date) pairs so no duplicates are added.

In [4]:
history_df = pd.read_parquet(HISTORY_PARQUET)
history_df["date"] = history_df["date"].dt.strftime("%Y-%m-%d")

history_df[["address", "chain", "date", "tvl_usd", "apy",
             "apy_base", "apy_base_7d", "il_7d"]].to_sql(
    "pool_history", con, if_exists="replace", index=False
)
con.commit()

count      = cur.execute("SELECT COUNT(*) FROM pool_history").fetchone()[0]
date_range = cur.execute("SELECT MIN(date), MAX(date) FROM pool_history").fetchone()
chains     = cur.execute("SELECT DISTINCT chain FROM pool_history ORDER BY chain").fetchall()
print(f"pool_history table: {count:,} rows")
print(f"Date range        : {date_range[0]}  ->  {date_range[1]}")
print(f"Chains            : {', '.join(r[0] for r in chains)}")

pool_history table: 36,994 rows
Date range        : 2022-03-27  ->  2026-03-26
Chains            : Arbitrum, Base, Ethereum, Polygon


## 4. Example Queries

In [5]:
# Top pools by average TVL across all history
print("Top pools by average historical TVL")
pd.read_sql("""
    SELECT p.chain, p.token0, p.token1, p.fee_tier,
           ROUND(AVG(h.tvl_usd) / 1e6, 2) AS avg_tvl_usd_m,
           ROUND(MAX(h.tvl_usd) / 1e6, 2) AS peak_tvl_usd_m,
           COUNT(h.date)                   AS days_tracked
    FROM pool_history h
    JOIN pools p ON h.address = p.address AND h.chain = p.chain
    GROUP BY h.address, h.chain
    ORDER BY avg_tvl_usd_m DESC
""", con)

Top pools by average historical TVL


,chain,token0,token1,fee_tier,avg_tvl_usd_m,peak_tvl_usd_m,days_tracked
0,Ethereum,WBTC,WETH,3000,187.72,434.86,1453
1,Ethereum,USDC,WETH,500,163.57,345.28,1433
2,Ethereum,USDC,WETH,3000,111.72,425.41,1433
3,Ethereum,WETH,USDT,3000,84.25,223.29,1453
4,Ethereum,WBTC,WETH,500,69.87,142.31,1453
5,Ethereum,USDC,USDT,100,62.69,222.27,1433
6,Ethereum,WBTC,USDC,3000,59.46,174.63,1433
7,Ethereum,WHITE,WETH,100,45.70,81.26,826
8,Ethereum,WBTC,CBBTC,100,43.84,69.92,528
9,Arbitrum,WBTC,WETH,500,30.34,79.91,1397


In [6]:
# Average base APY by pool — last 90 days
print("Average base APY — last 90 days")
pd.read_sql("""
    SELECT p.chain, p.token0, p.token1, p.fee_tier,
           ROUND(AVG(h.apy_base), 2) AS avg_apy_base_pct,
           ROUND(MAX(h.apy_base), 2) AS max_apy_base_pct
    FROM pool_history h
    JOIN pools p ON h.address = p.address AND h.chain = p.chain
    WHERE h.date >= date('now', '-90 days')
      AND h.apy_base IS NOT NULL
    GROUP BY h.address, h.chain
    ORDER BY avg_apy_base_pct DESC
""", con)

Average base APY — last 90 days


,chain,token0,token1,fee_tier,avg_apy_base_pct,max_apy_base_pct
0,Base,USDC,CBBTC,3000,53.73,579.90
1,Ethereum,WETH,USDT,3000,48.22,408.86
2,Base,WETH,CBBTC,3000,43.48,528.47
3,Base,USDC,CBBTC,500,41.73,202.90
4,Ethereum,USDC,WETH,500,38.99,345.15
5,Ethereum,WBTC,USDT,500,28.06,137.94
6,Ethereum,USDC,WETH,3000,27.12,193.45
7,Base,USDC,MAG7.SSI,3000,25.30,172.04
8,Base,DEFI.SSI,USDC,3000,21.27,87.67
9,Arbitrum,WBTC,WETH,500,21.05,118.74


In [7]:
# Monthly average TVL for top 5 pools
print("Monthly average TVL — top 5 pools")
pd.read_sql("""
    SELECT p.chain, p.token0 || '/' || p.token1 AS pair,
           strftime('%Y-%m', h.date)             AS month,
           ROUND(AVG(h.tvl_usd) / 1e6, 2)       AS avg_tvl_usd_m
    FROM pool_history h
    JOIN pools p ON h.address = p.address AND h.chain = p.chain
    WHERE (p.address, p.chain) IN (
        SELECT address, chain FROM pools ORDER BY tvl_usd DESC LIMIT 5
    )
    GROUP BY p.address, p.chain, month
    ORDER BY month DESC, avg_tvl_usd_m DESC
    LIMIT 30
""", con)

Monthly average TVL — top 5 pools


,chain,pair,month,avg_tvl_usd_m
0,Ethereum,USDC/WETH,2026-03,94.26
1,Ethereum,WETH/USDT,2026-03,61.55
2,Arbitrum,WBTC/WETH,2026-03,55.47
3,Ethereum,WBTC/WETH,2026-03,47.67
4,Ethereum,WBTC/WETH,2026-03,46.55
5,Ethereum,USDC/WETH,2026-02,78.85
6,Ethereum,WETH/USDT,2026-02,63.97
7,Ethereum,WBTC/WETH,2026-02,50.71
8,Arbitrum,WBTC/WETH,2026-02,48.89
9,Ethereum,WBTC/WETH,2026-02,37.90


In [8]:
con.close()
print(f"Database saved to {DB_PATH}  ({DB_PATH.stat().st_size / 1024:.0f} KB)")

Database saved to data\defi_pools.db  (3640 KB)
